# Lab 5: Fine-tuning a coding agent with SFT

## Notebook 3: Evaluate with execution-based pass@1

For code, the honest question is not "does the output look right" but "does it run and
produce the right answer". `pass@1` answers exactly that: generate one solution per task,
execute it against the task's unit tests, and report the fraction that pass.

We register a **custom scorer** (`codeexec_scorer.py`), a single self-contained file the
managed pipeline runs in its own container, the same mechanism Lab 1 uses for its
deterministic contract scorer. Here it executes the model's generated code against the
HumanEval unit tests, so the number is a real measurement rather than a judge's opinion.

Setting `evaluate_base_model=True` scores the **base and fine-tuned models in the same job**,
under identical conditions, which is what makes the before/after comparison fair.

> **Executing model-generated code.** The scorer runs untrusted code. In the managed
> pipeline it executes inside the evaluation container, isolated from your notebook, and each
> record runs in a subprocess with a wall-clock timeout. Adequate for a workshop; review the
> sandboxing before a larger run.

In [ ]:
%load_ext autoreload
%autoreload 2

#### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import time

from config import BASE_MODEL_ID, DATASET_PREFIX

base_model_id = BASE_MODEL_ID
BUCKET = bucket_name
sm = sm_client

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so every notebook derives the same name.
MAX_MPG_NAME_LENGTH = 63
suffix = "-coding-agent-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")

### The one switch

`RUN_EVALUATIONS = False` loads pre-computed results (fast, nothing billed). Set it to
`True` to launch the managed scorer yourself.

> **Pre-computed results are not published yet** for this lab. Until they are, set
> `RUN_EVALUATIONS = True` and run the scorer, or point `PRECOMPUTED_*` at your own S3 copy.

In [ ]:
RUN_EVALUATIONS = True

SCORER_OUTPUT = f"s3://{BUCKET}/coding-agent-scorer-eval"
print("RUN_EVALUATIONS =", RUN_EVALUATIONS)

### Register the pass@1 scorer

`codeexec_scorer.py` extracts the Python code from the model's output, runs it together with
the HumanEval `check(candidate)` tests in a sandboxed subprocess, and emits three metrics:
`pass_at_1` (the headline), `executes`, and `syntax_valid`.

The `get`-then-`create` shape makes the cell re-runnable.

In [ ]:
from sagemaker.ai_registry.air_constants import REWARD_FUNCTION
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.evaluator import Evaluator

SCORER_NAME = "code-pass-at-1-scorer"
try:
    scorer = Evaluator.get(name=SCORER_NAME)
    print("reusing registered scorer")
except Exception:
    scorer = Evaluator.create(name=SCORER_NAME, type=REWARD_FUNCTION,
                              source="codeexec_scorer.py", role=role,
                              sagemaker_session=sess, wait=True)
    scorer.refresh()
    print("registered scorer")

test_dataset = DataSet.get(name=f"{DATASET_PREFIX}-test")
resp = sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name, SortBy="CreationTime",
    SortOrder="Descending", MaxResults=1)
assert resp["ModelPackageSummaryList"], "no model packages found - run notebook 2 first"
model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"fine-tuned model: {model_package_arn}")

#### Sanity-check the scorer before registering (recommended)

Confirm the scorer passes correct code and fails wrong code, and that it reads
`model_response` (the model output) rather than `response` (the gold). Reading the gold
would score a perfect 1.0 on every record.

In [ ]:
import codeexec_scorer as CS

good = {"model_response": "```python\ndef add(a, b):\n    return a + b\n```",
        "entry_point": "add",
        "test": "def check(candidate):\n    assert candidate(2, 3) == 5\n",
        "task_id": "demo/0"}
bad = {**good, "model_response": "```python\ndef add(a, b):\n    return a - b\n```"}
leak = {**good, "model_response": "", "response": good["model_response"]}

for label, rec in [("correct", good), ("wrong", bad), ("empty (gold ignored)", leak)]:
    r = CS.score_record(rec)
    m = {x["name"]: x["value"] for x in r["metrics_list"]}
    print(f"{label:22s} pass_at_1={m['pass_at_1']} executes={m['executes']} syntax_valid={m['syntax_valid']}")

### Run base and fine-tuned in one job

`CustomScorerEvaluator` runs inference and scoring directly from the registered model
package. With `evaluate_base_model=True` it scores both models in the same job. No endpoint
is involved.

> **Generation budget.** `max_new_tokens = 1024` is comfortable for a single function.
> Qwen3 is a reasoning model, so if the base model spends its budget inside a `<think>` block
> its solution may be truncated, which shows up as low `syntax_valid`. If you see that, raise
> the budget or append `/no_think` in the prompt, keeping it identical for both models.

In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

if RUN_EVALUATIONS:
    scorer_eval = CustomScorerEvaluator(
        evaluator=scorer,
        dataset=test_dataset,
        model=model_package_arn,
        model_package_group=model_package_group_name,
        s3_output_path=SCORER_OUTPUT,
        evaluate_base_model=True,
        sagemaker_session=sess,
        role=role,
    )

    scorer_eval.hyperparameters.max_new_tokens = 1024
    scorer_eval.hyperparameters.max_model_len = 4096
    print("generation settings:",
          {k: v for k, v in scorer_eval.hyperparameters.to_dict().items()
           if k in ("max_new_tokens", "max_model_len", "temperature")})

    scorer_execution = scorer_eval.evaluate()
    print("launched:", scorer_execution.arn)

    while True:
        status = sm.describe_pipeline_execution(
            PipelineExecutionArn=scorer_execution.arn)["PipelineExecutionStatus"]
        print(f"{time.strftime('%H:%M:%S')}  {status}")
        if status != "Executing":
            break
        time.sleep(60)

### Read the results

The pipeline runs two parallel steps, `EvaluateBaseModel` and `EvaluateCustomModel`, so the
step name in the S3 key tells you which model a results file belongs to.

In [ ]:
import io
import json

from sagemaker.core import s3 as s3core

s3 = boto3.client("s3")


def read_results(prefix, step_marker):
    """Pull the aggregate results file the given pipeline step wrote."""
    keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(
                Bucket=BUCKET, Prefix=prefix)
            for o in page.get("Contents", [])
            if step_marker in o["Key"] and "/results_" in o["Key"] and o["Key"].endswith(".json")]
    if not keys:
        return None
    keys.sort()
    payload = json.loads(s3.get_object(Bucket=BUCKET, Key=keys[-1])["Body"].read())
    return list(payload["results"].values())[0]


prefix = SCORER_OUTPUT.split(f"{BUCKET}/")[1]
base_scored = read_results(prefix, "EvaluateBaseModel")
tuned_scored = read_results(prefix, "EvaluateCustomModel")

SHOW = ["pass_at_1", "executes", "syntax_valid"]
print(f"{'metric':16s} {'base':>9s} {'fine-tuned':>11s}")
for name in SHOW:
    b = f"{base_scored[name]*100:9.1f}" if base_scored and name in base_scored else f"{'-':>9s}"
    t = f"{tuned_scored[name]*100:11.1f}" if tuned_scored and name in tuned_scored else f"{'-':>11s}"
    print(f"{name:16s} {b} {t}")

The improvement to look for is a higher `pass_at_1` on the fine-tuned column, measured
identically to the base column. A fine-tune is not bit-reproducible on a 10K subset, so read
the direction and size of the gap, not the third decimal.

The pipeline also reports built-in ROUGE and BLEU. Ignore them here: string overlap with a
reference solution says little about whether code runs.

### Qualitative side-by-side (optional, great for a live demo)

Run a handful of fixed prompts through both models and print the outputs next to each other,
the quickest way to *show* the improvement in a room. This requires the endpoints/model from
notebook 4, or you can adapt it to call the model package directly. It is illustration, not
measurement: the `pass_at_1` table above is the number the lab reports.

In [ ]:
DEMO_PROMPTS = [
    "Write a Python function to find the longest palindromic substring in a string.",
    "Implement a binary search tree with insert, delete and search in Python.",
    "Write a function that returns the top 3 items by value from a list of dicts.",
    "Parse an ISO-8601 timestamp string and return the weekday name.",
    "Write a function to flatten an arbitrarily nested list of integers.",
]
for i, p in enumerate(DEMO_PROMPTS):
    print(f"[{i}] {p}")
print("\nRun these through the base and fine-tuned models (see notebook 4) and compare.")